In [1]:
import os
import json
from pathlib import Path

import torch


libgomp: Invalid value for environment variable OMP_NUM_THREADS
/home/jovyan/dnalm/my_saved_conda_envs/gena/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.chdir('/home/jovyan/dnalm/')

In [3]:
rmt_model_path = Path('/home/jovyan/dnalm/model_hub/rmt_bert_base_lastln_t2t_1000G_seglen_512_len_3992_maxnsegm_8_msz_10_bptt-1_bs256_lr_1e-05_wd_1e-04_fp16_O2/')

In [4]:
exp_config = json.load((rmt_model_path / 'config.json').open('r'))

In [5]:
for k in ['input_seq_len', 'model_cfg', 'model_cls', 'backbone_cls', 'input_size', 'num_mem_tokens', 'max_n_segments', 'tokenizer']:
    print(f'{k}: {exp_config[k]}')

input_seq_len: 3992
model_cfg: ./bert_configs/L12-H768-A12-V32k-preln-lastln.json
model_cls: modeling_rmt:RMTEncoderForMaskedLM
backbone_cls: modeling_bert:BertForMaskedLM
input_size: 512
num_mem_tokens: 10
max_n_segments: 8
tokenizer: /home/jovyan/dnalm/data/tokenizers/t2t_1000h_multi_32k


In [6]:
from transformers import AutoTokenizer, AutoConfig
tokenizer = AutoTokenizer.from_pretrained('./data/tokenizers/t2t_1000h_multi_32k/')

In [7]:
from src.gena_lm.modeling_bert import BertForTokenClassification

In [8]:
model_cfg = AutoConfig.from_pretrained('./data/configs/L12-H768-A12-V32k-preln-lastln.json')
model_cls = BertForTokenClassification
model = model_cls(config=model_cfg)

In [9]:
rmt_config = {
            'num_mem_tokens': exp_config['num_mem_tokens'],
            'max_n_segments': exp_config['max_n_segments'],
            'input_size': exp_config['input_size'],
            'bptt_depth': -1,
            'sum_loss': True,
            'tokenizer': tokenizer,
        }
from src.gena_lm.modeling_rmt import RMTEncoderForLetterLevelTokenClassification
rmt_cls = RMTEncoderForLetterLevelTokenClassification
model = rmt_cls(model, **rmt_config)

In [10]:
# load pre-trained weights
ckpt = torch.load(str(rmt_model_path / 'model_best.pth'), map_location='cpu')
missing_k, unexpected_k = model.load_state_dict(ckpt["model_state_dict"], strict=False)
print(f'missing: {missing_k}')
print(f'unexpected_k: {unexpected_k}')

missing: ['model.classifier.weight', 'model.classifier.bias']
unexpected_k: ['model.cls.predictions.bias', 'model.cls.predictions.transform.dense.weight', 'model.cls.predictions.transform.dense.bias', 'model.cls.predictions.transform.LayerNorm.weight', 'model.cls.predictions.transform.LayerNorm.bias', 'model.cls.predictions.decoder.weight', 'model.cls.predictions.decoder.bias']


In [11]:
model = model.eval()

In [12]:
pad_token_ids = {'input_ids': tokenizer.pad_token_id, 'token_type_ids': 0, 'attention_mask': 0}
pad_to_divisible_by = 1
# you can use it to build batches (mb remove code for labels)
def collate_fn(batch):
    feature_keys = ['input_ids', 'token_type_ids', 'attention_mask']
    padded_batch = {k: [] for k in feature_keys}
    max_seq_len = max([len(el['input_ids']) for el in batch])
    max_seq_len += (
        (pad_to_divisible_by - max_seq_len % pad_to_divisible_by)
        if max_seq_len % pad_to_divisible_by != 0
        else 0
    )
    for k in feature_keys:
        for i, el in enumerate(batch):
            padded_batch[k] += [
                np.concatenate(
                    [
                        batch[i][k],
                        np.array([pad_token_ids[k]] * max(0, max_seq_len - len(el[k])), dtype=np.int64),
                    ]
                )
            ]
    for k in padded_batch:
        padded_batch[k] = torch.from_numpy(np.stack(padded_batch[k]))
    padded_batch['labels'] = torch.tensor([el['labels'] for el in batch])
    return padded_batch


In [13]:
seq = 'ATGC'*1900
input_features = tokenizer([seq], return_tensors='pt')
input_features['input_ids'].shape

torch.Size([1, 952])

In [14]:
with torch.no_grad():
    out = model(**tokenizer([seq], return_tensors='pt'), output_hidden_states=True)

/home/jovyan/envs/py38_cu114/lib/python3.8/site-packages/transformers/modeling_utils.py:881: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


In [16]:
out.keys()

odict_keys(['logits', 'hidden_states', 'hidden_states_0', 'hidden_states_1', 'loss_0', 'loss_1', 'loss'])

In [ ]:
# 0 token is always CLS:
# segm 1: CLS MEM MEM MEM ... MEM SEQUENCE_part_1 SEP PAD PAD ...
# segm 2: CLS MEM MEM MEM ... MEM SEQUENCE_part_2 SEP PAD PAD ...
...

In [17]:
out['hidden_states_0'][-1].shape

torch.Size([1, 512, 768])

In [20]:
out['hidden_states_1'][-1].shape

torch.Size([1, 512, 768])